In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact
import numpy as np
import warnings
import statistics as stt
#import bamboolib as bam
#from brokenaxes import brokenaxes
import os
#import plotly.express as px

#  1. load all .exp files

In [2]:
def dirFiles(fDir, ending):
    fNames = []
    for i in os.listdir(fDir):
        if i.endswith(ending):
            fNames.append(i)
    return fNames

dataFiles = dirFiles('data/2022-02-15 Boron measurements', 'exp') # Here you can now exchange 'csv' for 'exp'

In [48]:
def outlierCorrection(measurement, factorSD):
    element_signal = np.array(measurement)
    mean = np.mean(element_signal, axis=0)
    sd = np.std(element_signal, axis=0)
    return [x for x in measurement if (x > mean - factorSD * sd) or (x < mean + factorSD * sd)]


def data_convert(data):
    data_float = []
    fil =  (data.str.contains('L|IC|C|H') == True)
    endnum = data[fil].index[0]
    for i in data[0:endnum]:
        data_float.append(float(i))
    return data_float

In [9]:
df = pd.read_csv('data/327_U.csv', header=22) 

# 2. Functions to calculate averages of one measurement point

# B11 background substact

In [68]:
def calculateSingleFile_B11(filename, bg_11B, factorSD):
#upload data:
    df = pd.read_csv('data/2022-02-15 Boron measurements/' + filename, sep='\t', header=22)  # Hi Jie, I already inserted the additional argument here
#calculate average signal and average backgroud:

    B11_data_float = data_convert(df['11B']) #conversion to float required, as numbers are stored as str
    fils = []
    filw = []
        
    for i in B11_data_float:
        if (i > bg_11B) == True:
            fils.append(i)
        else:
            filw.append(i)

    B_11_aver = np.mean(outlierCorrection(fils, factorSD))
    B_11_backg_aver = np.mean(outlierCorrection(filw, factorSD))
    res = B_11_aver - B_11_backg_aver
    return(res)

# B10 background substact for IC

In [69]:
def calculateSingleFile_B10_IC(filename, bg_10B, factorSD):
#upload data:
    df = pd.read_csv('data/2022-02-15 Boron measurements/' + filename, sep='\t', header=22)  # Hi Jie, I already inserted the additional argument here
#calculate average signal and average backgroud:

    B10_data_float = data_convert(df['10B']) #conversion to float required, as numbers are stored as str
    fils = []
    filw = []
        
    for i in B10_data_float:
        if (i > bg_10B) == True:
            fils.append(i)
        else:
            filw.append(i)

    B_10_aver = np.mean(outlierCorrection(fils, factorSD))
    B_10_backg_aver = np.mean(outlierCorrection(filw, factorSD))
    res = B_10_aver - B_10_backg_aver
    return(res)

# B10 background substact for FC

In [70]:
def calculateSingleFile_B10_FC(filename, bg_10B, factorSD):
#upload data:
    df = pd.read_csv('data/2022-02-15 Boron measurements/' + filename, sep='\t', header=22)  # Hi Jie, I already inserted the additional argument here
#calculate average signal and average backgroud:
    B10_data_float = data_convert(df['10B']) #conversion to float required, as numbers are stored as str
    B_interference_left = data_convert(df['9.9'])
    B_interference_right = data_convert(df['10.2'])
    
    B10_data = outlierCorrection(B10_data_float, 2)
    B_interference_left_res = outlierCorrection(B_interference_left, 2)
    B_interference_right_res = outlierCorrection(B_interference_right, 2)

    B10_interference = (B_interference_right_res - B_interference_left_res)*(10-9.9)/(10.2-9.9)
    
    fils = []
    filw = []    
    for i in B10_data:
        if (i > bg_10B) == True:
            fils.append(i)
        else:
            filw.append(i)
    B_10_backg_aver = np.mean(outlierCorrection(filw, factorSD))
    
    B_10_aver = np.mean(B10_data)
    
    res=B_10_aver-B_10_backg_aver-B10_interference
    return(res)

# 3. Calculate averages of all measured points

In [75]:
res = []
for i in dataFiles:
    res.append(calculateSingleFile_B11(i, 0.005, 2)/calculateSingleFile_B10_IC(i, 0.0005, 2))

In [76]:
res

[4.558845992535382,
 4.6962276736145645,
 4.7465798947205435,
 4.6343560834074315,
 4.716118633448838,
 4.724842501978045,
 4.710114168350337,
 4.745638259978176,
 4.706175626132932,
 4.830220028287914,
 4.63343788467471,
 4.660351286614246,
 5.245132200722873,
 4.618572104387733,
 4.6707659790384675,
 4.787160797019703,
 4.708776378459985,
 4.748056770762532,
 4.762496875662216,
 4.731671875417404,
 4.699920039691113,
 4.557548896378764,
 nan,
 4.632145153436631,
 4.708002975210631,
 4.858287062222151,
 4.808888891731603,
 4.5608167391620675,
 4.643605944414408,
 4.659399831619442,
 4.582969562008224,
 4.674550499568082,
 4.6689294682377085,
 4.643431826634492,
 nan,
 4.679040905948444,
 4.662461781425403,
 4.7057312154008315,
 4.689462770542566,
 4.5607469522988975,
 4.67533040996182,
 4.645731215784749,
 5.996101589346965,
 4.635775910675084,
 4.6827907926231385,
 4.696278149583466,
 4.507102952117413,
 4.707386226244053,
 4.695907384235818,
 4.635649423395761,
 nan,
 4.600193151331

### Some old code

In [ ]:
#check for outliers for signal:
    element_signal = np.array(fils)
    mean = np.mean(element_signal, axis=0)
    sd = np.std(element_signal, axis=0)
    final_list_signal = [x for x in fils if (x > mean - factorSD * sd)]
    final_list_signal = [x for x in final_list_signal if (x < mean + factorSD * sd)]
    
#check for outliers for background:
    element_backg = np.array(filw)
    mean = np.mean(element_backg, axis=0)
    sd = np.std(element_backg, axis=0)
    final_list_backg = [x for x in filw if (x > mean - factorSD * sd)]
    final_list_backg = [x for x in final_list_backg if (x < mean + factorSD * sd)]
    

def calculateSingleFile(filename, bg_10B, factorSD):
#upload data:
    df = pd.read_csv('data/2022-02-15 Boron measurements/' + filename, sep='\t', header=22)  # Hi Jie, I already inserted the additional argument here
#calculate average signal and average backgroud:
    endnum = df['10B'][df['10B'] == 'L3'].index[0]
#    B_10 = []
    fils = []
    filw = []
#    for i in df['10B'][0:endnum]:
#        B_10.append(float(i)) #conversion to float required, as numbers are stored as str
#    for i in B_10:
    for i in df['10B'][0:endnum]:
        meas = float(i)
        if (meas > bg_10B) == True:
            fils.append(meas)
        else:
            filw.append(meas)

#    outlier_corrected_signal =outlierCorrection(fils, factorSD)
#    outlier_corrected_backg =outlierCorrection(filw, factorSD)

#    B_10_aver = sum(outlier_corrected_signal )/len(outlier_corrected_signal )
#    B_10_backg_aver = sum(outlier_corrected_backg)/len(outlier_corrected_backg)

    B_10_aver = np.mean(outlierCorrection(fils, factorSD))
    B_10_backg_aver = np.mean(outlierCorrection(filw, factorSD))
    
    return(B_10_aver, B_10_backg_aver)